In [106]:
import os, re, requests, math
from dotenv import load_dotenv
from openai import AzureOpenAI
import json, time, yaml
from pathlib import Path

CONFIG_PATH = Path("config.yaml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

load_dotenv(config["paths"]["load_dotenv"])
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_API_BASE"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)
DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]  # deployment name, NOT the model name

In [107]:
def llm(messages, temperature=0):
    """One call to the model. Takes a list of messages, returns the reply text.
    """
    resp = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

In [108]:
def get_data(url, **params):
    resp = requests.get(url, params=params, headers={"User-Agent": config["http"]["user_agent"]},
                        timeout=config["http"]["timeout"])
    resp.raise_for_status()
    return resp.json()

In [109]:
# WORKING MEMORY
class WorkingMemory:
    def __init__(self, system):
        self.messages = [{"role": "system", "content": system}]
    def add(self, role, content):
        self.messages.append({"role": role, "content": content}) # append

In [110]:
# EPISODIC MEMORY
# timestamped, recalled by keywords
class EpisodicMemory:
    def __init__(self, path="episodes.json"):
        self.path = Path(path)
        self.episodes = json.loads(self.path.read_text()) if self.path.exists() else []
    def record(self, summary):
        self.episodes.append({"time": time.strftime("%Y-%m-%d %H:%M"), "event": summary})
        self.path.write_text(json.dumps(self.episodes, indent=2))
    def recall(self, keyword):
        return [e for e in self.episodes if keyword.lower() in e["event"].lower()]

In [111]:
# SEMANTIC MEMORY
# facts as key: value
class SemanticMemory:
    def __init__(self):
        self.facts = {}
    def learn(self, key, value):
        self.facts[key] = value
    def as_prompt(self):
        return "Known facts:\n" + "\n".join(f"{k}: {v}" for k, v in self.facts.items())

In [112]:
# PROCEDURAL MEMORY
# reusable instructions for how to behave or act
procedures = {
    "report_current_state": (
        "Step 1: Parse the user's location input and map it to its corresponding Singapore region "
        "('north', 'south', 'east', 'west', or 'central'). If the location is ambiguous or missing, "
        "default to the system's current location context or explicitly ask the user for clarification.\n"
        "Step 2: Concurrently or sequentially call the external API tools to gather the live Weather, "
        "Temperature, and 24-hour PSI/PM2.5 metrics for that specific region. If any critical API fails "
        "or times out, record a fallback failure state instead of inventing placeholder metrics.\n"
        "Step 3: Aggregate the data and evaluate it against standard safety thresholds (e.g., NEA PSI "
        "categories and heat stress rules). Synthesize a structural response containing the location "
        "timestamp, explicit readings, and a clear, deterministic activity recommendation based on the "
        "highest environmental risk factor found."
    ),
}

# Tool Functions
- Weather
- Temperature
- PSI readings

In [113]:
def fetch_psi() -> tuple[str, dict]:
    """Return (updated timestamp, {region: 24h PSI}) from NEA's real-time API."""
    item = get_data(config["urls"]["psi"])["data"]["items"][0]
    return item["updatedTimestamp"], item["readings"]["psi_twenty_four_hourly"]

def get_psi_readings() -> str:
    """Singapore's current 24-hour PSI by region (NEA). Current data, not a forecast. No arguments."""
    ts, psi = fetch_psi()
    return f"PSI for Singapore, updated {ts}:\n" + "\n".join(
        f"{r.capitalize()}: {v}" for r, v in psi.items())

In [114]:
print(get_psi_readings()[:200], "...\n")

PSI for Singapore, updated 2026-10-18T02:30:36+08:00:
North: 130
South: 170
West: 221
East: 181
Central: 184 ...



In [115]:
REGION_CENTRES = {r: tuple(c) for r, c in config["region_centres"].items()}
REGIONS = list(REGION_CENTRES)

def geocode(place: str, n: int=config["agent"]["n_candidates"]) -> list[dict]:
    """Top-n OneMap matches for a Singapore place: [{name, address, lat, lon}, ...]."""
    results = get_data(config["urls"]["onemap"], searchVal=place, returnGeom="Y", getAddrDetails="Y", pageNum=1)["results"]
    if not results:
        raise ValueError(f"No location found for {place!r}")
    return [{"name": r["SEARCHVAL"], "address": r["ADDRESS"], "lat": float(r["LATITUDE"]), "lon": float(r["LONGITUDE"])} 
            for r in results[:n]]

def nearest_region(lat: float, lon: float) -> str:
    return min(REGION_CENTRES, key=lambda r: math.dist((lat, lon), REGION_CENTRES[r]))

def psi_for_place(place: str) -> str:
    top = geocode(place)[0]
    lat, lon = top["lat"], top["lon"]
    region = nearest_region(lat, lon)
    ts, psi = fetch_psi()
    return f"{place} ({lat:.4f}, {lon:.4f}) -> {region} region: 24h PSI {psi[region]} (updated {ts})"

ntu = geocode("NTU")   # one OneMap call, reused below (avoids 429 Too Many Requests)
print(ntu)
print(nearest_region(ntu[0]["lat"], ntu[0]["lon"]))

[{'name': 'NANYANG TECHNOLOGICAL UNIVERSITY', 'address': '94 NANYANG CRESCENT NANYANG TECHNOLOGICAL UNIVERSITY SINGAPORE 637034', 'lat': 1.352949370203277, 'lon': 103.6892235086704}, {'name': 'NANYANG TECHNOLOGICAL UNIVERSITY', 'address': '71 NANYANG CRESCENT NANYANG TECHNOLOGICAL UNIVERSITY SINGAPORE 637035', 'lat': 1.354282743739479, 'lon': 103.6894395003443}, {'name': 'NANYANG TECHNOLOGICAL UNIVERSITY', 'address': '50 NANYANG AVENUE NANYANG TECHNOLOGICAL UNIVERSITY SINGAPORE 639798', 'lat': 1.343058936358113, 'lon': 103.6809726894903}, {'name': 'NTU INNOVATION CENTRE', 'address': '71 NANYANG DRIVE NTU INNOVATION CENTRE SINGAPORE 638075', 'lat': 1.348278894770232, 'lon': 103.6816475606479}, {'name': 'NTU@ONE-NORTH', 'address': '11 SLIM BARRACKS RISE NTU@ONE-NORTH SINGAPORE 138664', 'lat': 1.30376708827617, 'lon': 103.7899076195128}]
west


In [124]:
# resolving agent for geocode
def resolve_place(qns):
    place = llm([
        {"role": "system", "content": "Extract the Singapore place the user is asking about. Reply with only the place name."},
        {"role": "user", "content": qns},
    ]).strip()
    cands = geocode(place)   # raises ValueError if OneMap finds nothing
    listing = "\n".join(f"{i}: {c['name']} | {c['address']}" for i, c in enumerate(cands))

    reply = llm([
        {"role": "system", "content": "You match a place name to a list of map search results, so we can get coordinates for a weather and air-quality lookup. "
                                      "You are NOT answering the user's question; only decide which result is in the place they mean. "
                                      "A result matches if it IS that place, or is a building or location INSIDE that place "
                                      "(e.g. for a town or district, any location in that town counts). "
                                      "A result does NOT match if it only shares part of the name but is a different place. "
                                      "Reply with only the number of the first matching result, or -1 if none match."},
        {"role": "user", "content": f"Question: {qns}\nPlace: {place}\nCandidates:\n{listing}"},
    ]).strip()

    i = int(reply) if reply.lstrip("-").isdigit() else -1 # guardrail
    if not 0 <= i < len(cands):
        raise ValueError(f"No candidate matched {place!r}; picker replied {reply!r}")
    return {**cands[i], "query": place, "region": nearest_region(cands[i]["lat"], cands[i]["lon"])}

# verification
def verify_place(qns, p):
    reply = llm([
        {"role": "system", "content": "You check a location lookup for a weather and air-quality report, which only needs coordinates in the right area. "
                                      "Reply YES if the resolved location is the place the user asked about, or is a building or location INSIDE it "
                                      "(e.g. any location in a town counts for that town). "
                                      "Reply NO if it is a different place, even if the name is similar (e.g. NTUC instead of NTU), or is in a different area. "
                                      "Start with YES or NO, then a short reason."},
        {"role": "user", "content": f"Question: {qns}\nResolved to: {p['name']}, {p['address']}\n"
                                    "Is this the place the user meant?"},
    ])
    return reply.strip().upper().startswith("YES"), reply

In [117]:
PSI_BANDS = config["psi_bands"]   # {band: upper bound}, in order from config.yaml
def psi_band(v):
    return next(band for band, hi in PSI_BANDS.items() if v <= hi)

In [118]:
def fetch_temperature(lat, lon):
    d = get_data(config["urls"]["temperature"])["data"]
    stations = {s["id"]: (s["location"]["latitude"], s["location"]["longitude"]) for s in d["stations"]}
    reading = d["readings"][0]
    values = {r["stationId"]: r["value"] for r in reading["data"]}
    nearest = min(values, key=lambda s: math.dist((lat, lon), stations[s]))
    return values[nearest], round(sum(values.values()) / len(values), 1), reading["timestamp"]

In [119]:
def fetch_forecast(lat, lon):
    d = get_data(config["urls"]["forecast"])["data"]
    areas = {a["name"]: (a["label_location"]["latitude"], a["label_location"]["longitude"]) for a in d["area_metadata"]}
    item = d["items"][0]
    fc = {f["area"]: f["forecast"] for f in item["forecasts"]}
    area = min(fc, key=lambda a: math.dist((lat, lon), areas[a]))
    return area, fc[area], item["valid_period"]["text"]

In [120]:
def recommend(psi, temp, forecast):
    # deterministic: highest risk wins
    if psi > 200: return "Avoid outdoor activity; stay indoors."
    if psi > 100: return "Reduce prolonged or strenuous outdoor activity."
    if any(w in forecast.lower() for w in ("thunder", "rain", "showers")):
        return "Keep activities indoors or bring rain cover."
    if temp >= 33: return "Limit strenuous activity in the heat and stay hydrated."
    return "Normal activities are fine."

def gather(p):
    ts, psi = fetch_psi()
    temp, island_avg, _ = fetch_temperature(p["lat"], p["lon"])
    area, forecast, period = fetch_forecast(p["lat"], p["lon"])
    v = psi[p["region"]]
    return {"location": p["name"], "region": p["region"], "updated": ts,
            "psi_24h": v, "psi_band": psi_band(v),
            "temp_c": temp, "island_avg_c": island_avg, "temp_diff_c": round(temp - island_avg, 1),
            "forecast_area": area, "forecast": forecast, "forecast_period": period,
            "recommendation": recommend(v, temp, forecast)}

In [127]:
# report generation and self-checking
MAX_TRIES = config["agent"]["max_tries"]   # 1 first draft + up to (MAX_TRIES - 1) rewrites

class ReportCheckError(Exception):
    """Raised when the writer can't produce a report that matches the facts within MAX_TRIES."""
    def __init__(self, problems, draft):
        super().__init__("Report still failed checks: " + "; ".join(problems))
        self.problems = problems   # what was still wrong
        self.draft = draft         # last draft, kept for debugging

def write_report(facts, draft=None, problems=None):
    msgs = [
        {"role": "system", "content": procedures["report_current_state"] +
            "\nUse exactly this format, one item per line:\n"
            "Last updated: <time> for <location> (<region> region)\n"
            "PSI (24-hr): <value> (<band>)\n"
            "Temperature: <temp>°C (<diff>°C higher/lower than island average)\n"
            "Forecast (<period>, <area>): <forecast>\n"
            "Recommendation: <recommendation>"},
        {"role": "user", "content": json.dumps(facts)},
    ]
    if problems:
        # show the model its own draft, so it fixes it instead of starting from scratch
        msgs.append({"role": "assistant", "content": draft})
        msgs.append({"role": "user", "content": "Your draft above had these problems: " + "; ".join(problems)
                                                + ". Reply with the full corrected report."})
    return llm(msgs)

def has_number(text, value):
    """True if value appears as a whole number in text, so 21 doesn't match inside 218."""
    forms = {str(value)}
    if isinstance(value, float) and value.is_integer():
        forms.add(str(int(value)))   # 27.0 may be written as 27
    return any(re.search(rf"(?<![\d.]){re.escape(f)}(?!\.?\d)", text) for f in forms)

def check_report(report, facts):
    problems = []
    for key in ("psi_24h", "temp_c"):
        if not has_number(report, facts[key]):
            problems.append(f"{key} should be {facts[key]}")
    if not has_number(report, abs(facts["temp_diff_c"])):   # abs: "-1.2" or "1.2 below" both fine
        problems.append(f"temp_diff_c should be {facts['temp_diff_c']}")
    # band in brackets, as in the template, so "unhealthy" doesn't match inside "very unhealthy"
    if f"({facts['psi_band']})" not in report.lower():
        problems.append(f"psi_band should be {facts['psi_band']!r}")
    for key in ("forecast", "recommendation"):
        if facts[key].lower() not in report.lower():
            problems.append(f"missing {key}: {facts[key]!r}")
    return problems

def write_checked_report(facts, max_tries=MAX_TRIES):
    """Write -> check -> rewrite with feedback, until the report passes or tries run out."""
    report, problems = None, None
    for _ in range(max_tries):
        report = write_report(facts, report, problems)
        problems = check_report(report, facts)
        if not problems:
            return report
    raise ReportCheckError(problems, report)

def answer(qns):
    p = resolve_place(qns) # obtain the results from the given question
    ok, why = verify_place(qns, p) # cross-check the returned results from facts
    if not ok:
        return f"I'm not sure which place you mean ({p['name']}?). {why}"
    facts = gather(p)
    return write_checked_report(facts)


In [129]:
qns = "What is the status like in tampines"
# location mapping resolve
p = resolve_place(qns)
print(p)
# verification
ok, why = verify_place(qns, p)
print(ok, "|", why)
# fact check against return results
facts = gather(p)
print(json.dumps(facts, indent=2))
# report
report = write_checked_report(facts) 
print("report:\n" + report)
print("  check:", check_report(report, facts) or "passed")


{'name': 'TAMPINES 9 BUILDING', 'address': '9 TAMPINES STREET 92 TAMPINES 9 BUILDING SINGAPORE 528871', 'lat': 1.343330328726356, 'lon': 103.9443723970848, 'query': 'Tampines', 'region': 'east'}
True | YES, the resolved location is within Tampines, so it matches the user's query about Tampines.
{
  "location": "TAMPINES 9 BUILDING",
  "region": "east",
  "updated": "2026-10-18T03:00:35+08:00",
  "psi_24h": 180,
  "psi_band": "unhealthy",
  "temp_c": 28.1,
  "island_avg_c": 27.3,
  "temp_diff_c": 0.8,
  "forecast_area": "Tampines",
  "forecast": "Cloudy",
  "forecast_period": "2.30 am to 4.30 am",
  "recommendation": "Reduce prolonged or strenuous outdoor activity."
}
report:
Last updated: 2026-10-18T03:00:35+08:00 for TAMPINES 9 BUILDING (east region)
PSI (24-hr): 180 (unhealthy)
Temperature: 28.1°C (0.8°C higher than island average)
Forecast (2.30 am to 4.30 am, Tampines): Cloudy
Recommendation: Reduce prolonged or strenuous outdoor activity.
  check: passed
